# Нейросетевой синтез четырёхплоскостного ДОЭ

Задание хакатона: 4 одноканальных PNG 128×128 → фазовый ДОЭ 128×128 →
4 реконструкции методом Френеля; λ=532 нм, пиксель ПВМС 10 мкм,
z=[24.1,25.3,26.5,27.7] мм. Экспорт: 8-битные PNG, Pearson и SSIM.

**Run All** запускает обучение, тестовую оценку и экспорт. Включите
Settings → Accelerator → GPU T4 ×2 при доступной квоте; код также работает на CPU.
Интернет и Kaggle API-токен внутри notebook не требуются.

В документах нет четырёх исходных объектных PNG: по умолчанию используется явно
обозначенный процедурный демонстрационный набор. Для своих данных укажите четыре
пути в `INPUT_IMAGES` ниже. Их порядок соответствует возрастающему z.

Метод — фиксированная развёрнутая нейросеть с оптическими проекциями и обучаемыми
CNN-коррекциями. Обучение без эталонных голограмм; при инференсе нет шагов оптимизатора.
Тестовые изображения не участвуют в обучении и выборе checkpoint.


In [ ]:
# Embedded source: inspectable, no Git clone or external download.
from pathlib import Path
import json, os, sys
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'notebook-workdir'
PROJECT = BASE / 'nuclear-hack-source'
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCE_FILES = {'doe/__init__.py': '"""Four-plane neural diffractive optical element synthesis."""\n\n__version__ = "0.1.0"\n', 'doe/__main__.py': 'from .cli import main\n\nmain()\n', 'doe/cli.py': 'import argparse\nimport json\nfrom pathlib import Path\n\nfrom .data import generate_dataset\nfrom .engine import TrainConfig, evaluate, predict, train\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description="Four-plane neural DOE synthesis, single-FFT Fresnel optics")\n    sub = parser.add_subparsers(dest="command", required=True)\n    generate = sub.add_parser("generate", help="Generate independent procedural train/validation/test quartets")\n    generate.add_argument("--output", required=True)\n    generate.add_argument("--train", type=int, default=128)\n    generate.add_argument("--validation", type=int, default=24)\n    generate.add_argument("--test", type=int, default=24)\n    generate.add_argument("--seed", type=int, default=2026)\n    fit = sub.add_parser("train", help="Train with differentiable Fresnel reconstruction, without DOE labels")\n    fit.add_argument("--data", required=True)\n    fit.add_argument("--output", required=True)\n    for flag, kind, default in (("epochs", int, 24), ("batch-size", int, 4), ("stages", int, 6),\n                                ("width", int, 12), ("learning-rate", float, 0.005), ("seed", int, 42)):\n        fit.add_argument("--" + flag, type=kind, default=default)\n    infer = sub.add_parser("predict", help="Export DOE, four amplitude PNGs and metrics for exactly four input PNGs")\n    infer.add_argument("--checkpoint", required=True)\n    infer.add_argument("--images", nargs=4, required=True, metavar="PNG")\n    infer.add_argument("--output", required=True)\n    assess = sub.add_parser("evaluate", help="Held-out comparison against untrained network and GS references")\n    assess.add_argument("--data", required=True)\n    assess.add_argument("--checkpoint", required=True)\n    assess.add_argument("--output", required=True)\n    assess.add_argument("--batch-size", type=int, default=4)\n    assess.add_argument("--gs-steps", type=int, default=80)\n    for p in (fit, infer, assess):\n        p.add_argument("--device", default="auto")\n        p.add_argument("--threads", type=int, default=2)\n    args = vars(parser.parse_args(argv))\n    command = args.pop("command")\n    if command == "generate":\n        generate_dataset(args.pop("output"), **args)\n    elif command == "train":\n        data, output = args.pop("data"), args.pop("output")\n        train(data, output, TrainConfig(**args))\n    elif command == "predict":\n        report = predict(**args)\n        print(json.dumps({"mean_pearson": report["mean_pearson"], "mean_ssim": report["mean_ssim"]}, indent=2))\n    else:\n        evaluate(**args)\n\n\nif __name__ == "__main__":\n    main()\n', 'doe/data.py': '"""Procedural, seed-separated image quartets; no external dataset is required."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFilter, ImageFont\nimport torch\nfrom torch.utils.data import Dataset\n\n\ndef image_array(path, size=128):\n    with Image.open(path) as image:\n        if image.format != "PNG" or image.mode != "L" or image.size != (size, size):\n            raise ValueError(f"{path}: expected an 8-bit grayscale PNG ({size}x{size}), got {image.format}/{image.mode}/{image.size}")\n        return np.asarray(image, dtype=np.uint8).copy()\n\n\ndef load_quartet(paths, size=128):\n    if len(paths) != 4:\n        raise ValueError("supply exactly four PNGs in increasing distance order")\n    arrays = np.stack([image_array(path, size) for path in paths])\n    if (arrays.max(axis=(-2, -1)) == 0).any():\n        raise ValueError("completely black target planes are not supported")\n    return torch.from_numpy(arrays.astype(np.float32) / 255)[None]\n\n\ndef procedural_image(seed, size=128):\n    rng = np.random.default_rng(seed)\n    scale = 2\n    image = Image.new("L", (size * scale, size * scale), 0)\n    draw = ImageDraw.Draw(image)\n    kind = int(rng.integers(0, 4))\n    if kind == 0:\n        font = ImageFont.load_default(size=int(rng.integers(38, 80)) * scale)\n        text = str(rng.integers(0, 10))\n        box = draw.textbbox((0, 0), text, font=font)\n        w, h = box[2] - box[0], box[3] - box[1]\n        x = (size * scale - w) // 2 + int(rng.integers(-12, 13)) * scale\n        y = (size * scale - h) // 2 - box[1] + int(rng.integers(-12, 13)) * scale\n        draw.text((x, y), text, font=font, fill=255)\n    else:\n        for _ in range(int(rng.integers(1, 5))):\n            x, y = rng.integers(size // 8, size * 3 // 4, 2)\n            w, h = rng.integers(size // 8, size // 2, 2)\n            bounds = tuple(int(v * scale) for v in (x, y, min(x + w, size - 5), min(y + h, size - 5)))\n            value = int(rng.integers(140, 256))\n            if kind == 1:\n                draw.ellipse(bounds, fill=value)\n            elif kind == 2:\n                draw.rectangle(bounds, fill=value)\n            else:\n                draw.ellipse(bounds, outline=value, width=int(rng.integers(3, 8)) * scale)\n        if kind == 3:\n            draw.line([(20 * scale, 105 * scale), (100 * scale, 30 * scale)], fill=220, width=3 * scale)\n    image = image.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.5, 1.6)) * scale))\n    image = image.resize((size, size), Image.Resampling.LANCZOS)\n    array = np.array(image)\n    array = np.rint(array.astype(np.float64) * (255 / max(int(array.max()), 1))).astype(np.uint8)\n    return array\n\n\ndef generate_dataset(root, train=128, validation=24, test=24, seed=2026):\n    root = Path(root)\n    if root.exists() and any(root.iterdir()):\n        raise FileExistsError(f"Refusing to overwrite existing dataset: {root}")\n    if min(train, validation, test) < 1:\n        raise ValueError("all three splits must be nonempty")\n    root.mkdir(parents=True, exist_ok=True)\n    manifest = {"source": "procedurally generated demonstration data, not official hackathon inputs",\n                "seed": seed, "size": 128, "mode": "L", "splits": {}}\n    for split_id, (split, count) in enumerate((("train", train), ("validation", validation), ("test", test))):\n        records = []\n        for index in range(count):\n            sample = root / split / f"sample_{index:04d}"\n            sample.mkdir(parents=True)\n            seeds = []\n            hashes = []\n            for plane in range(4):\n                image_seed = seed + split_id * 1_000_000 + index * 16 + plane * 101\n                path = sample / f"plane_{plane + 1}.png"\n                Image.fromarray(procedural_image(image_seed)).save(path)\n                seeds.append(image_seed)\n                hashes.append(hashlib.sha256(path.read_bytes()).hexdigest())\n            records.append({"path": str(sample.relative_to(root)), "image_seeds": seeds, "sha256": hashes})\n        manifest["splits"][split] = records\n    (root / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    return manifest\n\n\nclass QuartetDataset(Dataset):\n    def __init__(self, root, split):\n        root = Path(root)\n        manifest = json.loads((root / "manifest.json").read_text())\n        self.samples = [root / item["path"] for item in manifest["splits"][split]]\n        if not self.samples:\n            raise ValueError(f"empty dataset split: {split}")\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, index):\n        return load_quartet([self.samples[index] / f"plane_{i}.png" for i in range(1, 5)])[0]\n', 'doe/engine.py': '"""Reproducible training, held-out evaluation and inference."""\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom .data import QuartetDataset, load_quartet\nfrom .model import FresnelNet, gerchberg_saxton, physics_loss\nfrom .optics import OpticsConfig\nfrom .results import export_result, phase_metrics\n\n\n@dataclass\nclass TrainConfig:\n    epochs: int = 24\n    batch_size: int = 4\n    learning_rate: float = 0.005\n    stages: int = 6\n    width: int = 12\n    seed: int = 42\n    device: str = "auto"\n    threads: int = 2\n\n\ndef setup(seed=42, device="auto", threads=2):\n    if threads < 1:\n        raise ValueError("threads must be >= 1")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.set_num_threads(threads)\n    if device == "auto":\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n    if device.startswith("cuda") and not torch.cuda.is_available():\n        raise RuntimeError("CUDA requested but unavailable; select --device cpu")\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cudnn.deterministic = True\n    return torch.device(device)\n\n\ndef sync(device):\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n\n\ndef load_model(checkpoint, device):\n    state = torch.load(checkpoint, map_location=device, weights_only=True)\n    if state.get("format_version") != 1:\n        raise ValueError("unsupported checkpoint format")\n    optics = OpticsConfig(**state["optics"])\n    model = FresnelNet(optics, **state["architecture"]).to(device)\n    model.load_state_dict(state["model"])\n    model.eval()\n    return model, state\n\n\n@torch.no_grad()\ndef validation_loss(model, loader, device):\n    model.eval()\n    total, count = 0.0, 0\n    for targets in loader:\n        targets = targets.to(device)\n        loss = physics_loss(model.optics, model(targets), targets)\n        total += float(loss) * len(targets)\n        count += len(targets)\n    if not count:\n        raise ValueError("validation set is empty")\n    return total / count\n\n\ndef train(data, output, config=TrainConfig()):\n    if config.epochs < 1 or config.batch_size < 1 or config.learning_rate <= 0:\n        raise ValueError("epochs, batch size and learning rate must be positive")\n    device = setup(config.seed, config.device, config.threads)\n    output = Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(f"Refusing to overwrite training run: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    train_data, val_data = QuartetDataset(data, "train"), QuartetDataset(data, "validation")\n    loader = DataLoader(train_data, batch_size=config.batch_size, shuffle=True,\n                        generator=torch.Generator().manual_seed(config.seed), num_workers=0)\n    val_loader = DataLoader(val_data, batch_size=config.batch_size, num_workers=0)\n    model = FresnelNet(stages=config.stages, width=config.width).to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)\n    initial = validation_loss(model, val_loader, device)\n    initial_parameters = {name: p.detach().clone() for name, p in model.named_parameters()}\n    best_loss, best_epoch = initial, 0\n    history = [{"epoch": 0, "train_loss": None, "validation_loss": initial}]\n    manifest_hash = hashlib.sha256((Path(data) / "manifest.json").read_bytes()).hexdigest()\n\n    def save(path, epoch, val_loss):\n        torch.save({"format_version": 1, "model": model.state_dict(),\n                    "architecture": {"stages": config.stages, "width": config.width},\n                    "optics": model.optics.config.to_dict(), "training": asdict(config),\n                    "epoch": epoch, "validation_loss": val_loss,\n                    "dataset_manifest_sha256": manifest_hash}, path)\n\n    save(output / "initial.pt", 0, initial)\n    save(output / "best.pt", 0, initial)\n    start = time.perf_counter()\n    print(f"device={device}; train={len(train_data)} validation={len(val_data)}; initial_val={initial:.6f}", flush=True)\n    for epoch in range(1, config.epochs + 1):\n        model.train()\n        total = 0.0\n        for targets in loader:\n            targets = targets.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            phase = model(targets)\n            loss = physics_loss(model.optics, phase, targets)\n            if not torch.isfinite(loss):\n                raise FloatingPointError("nonfinite training loss")\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n            optimizer.step()\n            total += float(loss.detach()) * len(targets)\n        val_loss = validation_loss(model, val_loader, device)\n        if val_loss < best_loss:\n            best_loss, best_epoch = val_loss, epoch\n            save(output / "best.pt", epoch, val_loss)\n        record = {"epoch": epoch, "train_loss": total / len(train_data), "validation_loss": val_loss,\n                  "learning_rate": optimizer.param_groups[0]["lr"]}\n        history.append(record)\n        scheduler.step()\n        print(f"epoch={epoch:03d} train={record[\'train_loss\']:.6f} val={val_loss:.6f} best={best_loss:.6f}", flush=True)\n    sync(device)\n    save(output / "last.pt", config.epochs, history[-1]["validation_loss"])\n    drift = sum(float((p.detach() - initial_parameters[name]).square().sum()) for name, p in model.named_parameters())**0.5\n    report = {"config": asdict(config), "device": str(device), "torch": torch.__version__,\n              "train_count": len(train_data), "validation_count": len(val_data),\n              "selection": "minimum validation physics loss; test split never used for selection",\n              "initial_validation_loss": initial, "best_validation_loss": best_loss, "best_epoch": best_epoch,\n              "parameter_l2_change_last_vs_initial": drift,\n              "elapsed_seconds": time.perf_counter() - start, "history": history,\n              "dataset_manifest_sha256": manifest_hash}\n    (output / "training.json").write_text(json.dumps(report, indent=2, allow_nan=False) + "\\n")\n    return report\n\n\n@torch.no_grad()\ndef evaluate(data, checkpoint, output, device="auto", batch_size=4, gs_steps=80, threads=2):\n    device = setup(device=device, threads=threads)\n    model, state = load_model(checkpoint, device)\n    initial = FresnelNet(model.optics.config, stages=model.stages, width=model.width).to(device).eval()\n    # Identical analytical initialization; CNN starts as exact zero correction.\n    test = QuartetDataset(data, "test")\n    loader = DataLoader(test, batch_size=batch_size, num_workers=0)\n    methods = {"neural": lambda x: model(x), "untrained_network": lambda x: initial(x),\n               f"gs_{model.stages}": lambda x: gerchberg_saxton(model.optics, x, model.stages),\n               f"gs_{gs_steps}": lambda x: gerchberg_saxton(model.optics, x, gs_steps)}\n    records = {name: [] for name in methods}\n    seconds = {name: 0.0 for name in methods}\n    # Exclude one warmup batch per method from performance measurements.\n    warmup = next(iter(loader)).to(device)\n    for operation in methods.values():\n        operation(warmup)\n    for targets in loader:\n        targets = targets.to(device)\n        for name, operation in methods.items():\n            sync(device)\n            start = time.perf_counter()\n            phase = operation(targets)\n            sync(device)\n            seconds[name] += time.perf_counter() - start\n            records[name].extend(phase_metrics(model.optics, phase, targets))\n    summary = {}\n    for name, values in records.items():\n        correlations = [v["pearson"] for v in values if v["pearson"] is not None]\n        summary[name] = {"count": len(values), "mean_pearson": float(np.mean(correlations)) if correlations else None,\n                         "mean_ssim": float(np.mean([v["ssim"] for v in values])),\n                         "seconds_per_quartet_batched": seconds[name] / len(values)}\n    report = {"split": "test", "sample_count": len(test), "device": str(device), "batch_size": batch_size,\n              "checkpoint_epoch": state["epoch"], "checkpoint_sha256": hashlib.sha256(Path(checkpoint).read_bytes()).hexdigest(),\n              "quantization": "8-bit phase and 8-bit reconstructed amplitudes, same for all methods",\n              "timing": "one warmed pass per test batch; includes phase synthesis, excludes image IO and metric computation; batched throughput, not single-sample latency",\n              "summary": summary, "per_sample": records}\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    if output.exists():\n        raise FileExistsError(f"Refusing to overwrite evaluation: {output}")\n    output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\\n")\n    print(json.dumps(summary, indent=2), flush=True)\n    return report\n\n\n@torch.no_grad()\ndef predict(checkpoint, images, output, device="auto", threads=2, source="user PNGs"):\n    device = setup(device=device, threads=threads)\n    model, state = load_model(checkpoint, device)\n    targets = load_quartet(images).to(device)\n    model(targets)  # warmup\n    sync(device)\n    start = time.perf_counter()\n    phase = model(targets)\n    sync(device)\n    elapsed = time.perf_counter() - start\n    metadata = {"method": "FresnelNet", "network_stages": model.stages, "cnn_width": model.width,\n                "source": source, "device": str(device), "checkpoint_epoch": state["epoch"],\n                "checkpoint_sha256": hashlib.sha256(Path(checkpoint).read_bytes()).hexdigest(),\n                "network_forward_seconds": elapsed, "test_time_optimizer_steps": 0}\n    return export_result(output, model.optics, phase, images, metadata)\n', 'doe/model.py': '"""A fixed-depth, trainable, physics-informed unrolled network."""\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom .optics import Fresnel, OpticsConfig, normalize_targets, unit_complex\n\n\ndef initialization(optics, targets):\n    # Deterministic random output phases reduce coherent bias. Same seed at test time.\n    generator = torch.Generator().manual_seed(1701)\n    phase = (torch.rand((4, optics.config.size, optics.config.size), generator=generator) * 2 - 1) * math.pi\n    phase = phase.to(device=targets.device, dtype=targets.dtype)\n    fields = targets * torch.polar(torch.ones_like(phase), phase)\n    return unit_complex(optics.adjoint(fields).mean(dim=1))\n\n\nclass FresnelNet(nn.Module):\n    """Six-to-eight optical projection layers with learned CNN phase corrections.\n\n    Inference is one fixed-depth network forward pass, without optimizer steps.\n    This is an unrolled neural method, not a claim of zero Fourier transforms.\n    """\n\n    def __init__(self, config=OpticsConfig(), stages=6, width=12):\n        super().__init__()\n        if stages < 1 or width < 1:\n            raise ValueError("stages and width must be positive")\n        self.optics = Fresnel(config)\n        self.stages = stages\n        self.width = width\n        self.plane_logits = nn.Parameter(torch.zeros(stages, 4))\n        # Start at the classical projection (rho=1); learn under/over-relaxation in (0,2).\n        self.relax_logits = nn.Parameter(torch.zeros(stages))\n        self.corrector = nn.Sequential(\n            nn.Conv2d(4, width, 3, padding=1, padding_mode="circular"), nn.SiLU(),\n            nn.Conv2d(width, width, 3, padding=1, padding_mode="circular"), nn.SiLU(),\n            nn.Conv2d(width, 1, 1),\n        )\n        nn.init.zeros_(self.corrector[-1].weight)\n        nn.init.zeros_(self.corrector[-1].bias)\n\n    def forward(self, targets):\n        amplitude = normalize_targets(targets)\n        field = initialization(self.optics, amplitude)\n        for stage in range(self.stages):\n            reconstructed = self.optics.field(field)\n            constrained = amplitude * unit_complex(reconstructed)\n            back = self.optics.adjoint(constrained)\n            weights = self.plane_logits[stage].softmax(dim=0)[None, :, None, None]\n            projected = unit_complex((weights * back).sum(dim=1))\n            features = torch.stack((field.real, field.imag, projected.real, projected.imag), dim=1)\n            correction = 0.35 * torch.tanh(self.corrector(features)[:, 0])\n            relax = 2 * self.relax_logits[stage].sigmoid()\n            field = unit_complex((1 - relax) * field + relax * projected)\n            field = field * torch.polar(torch.ones_like(correction), correction)\n        return torch.angle(field)\n\n\n@torch.no_grad()\ndef gerchberg_saxton(optics, targets, steps=80):\n    """Multi-plane alternating-projection reference, using the identical physics."""\n    if steps < 1:\n        raise ValueError("steps must be positive")\n    amplitude = normalize_targets(targets)\n    field = initialization(optics, amplitude)\n    for _ in range(steps):\n        reconstructed = optics.field(field)\n        field = unit_complex(optics.adjoint(amplitude * unit_complex(reconstructed)).mean(dim=1))\n    return torch.angle(field)\n\n\ndef physics_loss(optics, phase, targets):\n    """Supervision-free amplitude loss; no ground-truth holograms are needed."""\n    desired = normalize_targets(targets)\n    amplitude = optics(phase).abs()\n    loss = F.mse_loss(amplitude, desired)\n    for scale in (2, 4):\n        loss = loss + 0.15 * F.mse_loss(F.avg_pool2d(amplitude, scale), F.avg_pool2d(desired, scale))\n    centered_a = amplitude - amplitude.mean(dim=(-2, -1), keepdim=True)\n    centered_b = desired - desired.mean(dim=(-2, -1), keepdim=True)\n    corr = (centered_a * centered_b).mean(dim=(-2, -1)) / (\n        centered_a.square().mean(dim=(-2, -1)) * centered_b.square().mean(dim=(-2, -1))\n    ).clamp_min(1e-10).sqrt()\n    return loss + 0.1 * (1 - corr).mean()\n', 'doe/optics.py': '"""Centered single-FFT Fresnel propagation on its natural output grids."""\nfrom dataclasses import asdict, dataclass\nimport math\n\nimport torch\nfrom torch import nn\n\n\n@dataclass(frozen=True)\nclass OpticsConfig:\n    size: int = 128\n    wavelength_m: float = 532e-9\n    pixel_pitch_m: float = 10e-6\n    distances_m: tuple[float, ...] = (0.0241, 0.0253, 0.0265, 0.0277)\n\n    def __post_init__(self):\n        if self.size < 8 or self.size % 2:\n            raise ValueError("size must be even and >= 8")\n        if not math.isfinite(self.wavelength_m) or self.wavelength_m <= 0:\n            raise ValueError("wavelength must be finite and positive")\n        if not math.isfinite(self.pixel_pitch_m) or self.pixel_pitch_m <= 0:\n            raise ValueError("pixel pitch must be finite and positive")\n        if len(self.distances_m) != 4 or any(not math.isfinite(z) or z <= 0 for z in self.distances_m):\n            raise ValueError("exactly four finite positive distances are required")\n\n    @property\n    def output_pixel_pitches_m(self):\n        return tuple(self.wavelength_m * z / (self.size * self.pixel_pitch_m) for z in self.distances_m)\n\n    def to_dict(self):\n        return asdict(self)\n\n\ndef fft2c(field):\n    return torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))\n\n\ndef ifft2c(field):\n    return torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(field, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))\n\n\nclass Fresnel(nn.Module):\n    """Unitary preconditioned Fresnel operator; physical() restores its prefactor.\n\n    Input: phase [B,H,W], uniform unit-amplitude illumination.\n    Output: complex field [B,4,H,W]. Each plane has its own output pixel pitch.\n    The unitary normalization changes only one constant per plane, not its image.\n    """\n\n    def __init__(self, config=OpticsConfig(), dtype=torch.float32):\n        super().__init__()\n        self.config = config\n        n, dx, lam = config.size, config.pixel_pitch_m, config.wavelength_m\n        # Compute oscillatory chirps in double precision before optional casting.\n        axis = (torch.arange(n, dtype=torch.float64) - n // 2) * dx\n        yy, xx = torch.meshgrid(axis, axis, indexing="ij")\n        z = torch.tensor(config.distances_m, dtype=torch.float64)[:, None, None]\n        input_phase = math.pi * (xx.square() + yy.square())[None] / (lam * z)\n        output_axis = (torch.arange(n, dtype=torch.float64) - n // 2)[None] * (lam * z[:, 0, 0] / (n * dx))[:, None]\n        output_phase = math.pi * (output_axis[:, :, None].square() + output_axis[:, None, :].square()) / (lam * z)\n        complex_dtype = torch.complex128 if dtype == torch.float64 else torch.complex64\n        self.register_buffer("input_chirp", torch.polar(torch.ones_like(input_phase), input_phase).to(complex_dtype))\n        self.register_buffer("output_chirp", torch.polar(torch.ones_like(output_phase), output_phase).to(complex_dtype))\n        global_phase = 2 * math.pi * z / lam - math.pi / 2\n        factor = n * dx**2 / (lam * z) * torch.polar(torch.ones_like(global_phase), global_phase)\n        self.register_buffer("physical_factor", factor.to(complex_dtype))\n\n    def field(self, field):\n        if field.shape[-2:] != (self.config.size, self.config.size):\n            raise ValueError("field size does not match optics configuration")\n        return self.output_chirp * fft2c(field[:, None] * self.input_chirp)\n\n    def forward(self, phase):\n        return self.field(torch.polar(torch.ones_like(phase), phase))\n\n    def adjoint(self, fields):\n        """Per-plane unitary adjoint, without summing over planes."""\n        return ifft2c(fields * self.output_chirp.conj()) * self.input_chirp.conj()\n\n    def physical(self, phase):\n        return self(phase) * self.physical_factor\n\n\ndef unit_complex(field, eps=1e-7):\n    # Stable fallback at zeros, including blank regions in target back-projection.\n    return torch.where(field.abs() > eps, field / field.abs().clamp_min(eps), torch.ones_like(field))\n\n\ndef normalize_targets(targets):\n    if targets.ndim != 4 or targets.shape[1] != 4:\n        raise ValueError("targets must have shape [batch,4,height,width]")\n    if not torch.isfinite(targets).all() or (targets < 0).any():\n        raise ValueError("targets must be finite nonnegative amplitudes")\n    rms = targets.square().mean(dim=(-2, -1), keepdim=True).sqrt()\n    if (rms <= 1e-8).any():\n        raise ValueError("a completely black plane is incompatible with unit-amplitude phase-only illumination")\n    return targets / rms\n', 'doe/results.py': '"""Quantized deliverables and independently computable image quality metrics."""\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom skimage.metrics import structural_similarity\nimport torch\n\nfrom .data import load_quartet\n\n\ndef encode_phase(phase):\n    phase = np.asarray(phase, dtype=np.float64)\n    if not np.isfinite(phase).all() or (np.abs(phase) > math.pi + 1e-6).any():\n        raise ValueError("phase must be finite and in [-pi, pi]")\n    return np.rint((np.clip(phase, -math.pi, math.pi) + math.pi) * (255 / (2 * math.pi))).astype(np.uint8)\n\n\ndef decode_phase(image):\n    image = np.asarray(image)\n    if image.dtype != np.uint8:\n        raise ValueError("phase PNG samples must be uint8")\n    return image.astype(np.float64) * (2 * math.pi / 255) - math.pi\n\n\ndef amplitude_png(amplitude):\n    amplitude = np.asarray(amplitude, dtype=np.float64)\n    if not np.isfinite(amplitude).all() or (amplitude < 0).any():\n        raise ValueError("amplitude must be finite and nonnegative")\n    peak = float(amplitude.max())\n    return np.rint(255 * amplitude / max(peak, 1e-30)).astype(np.uint8), peak\n\n\ndef metrics(target, reconstructed):\n    """Metrics of actual 8-bit images. No fitting, denoising or target-dependent gain."""\n    a = np.asarray(target, dtype=np.float64) / 255\n    b = np.asarray(reconstructed, dtype=np.float64) / 255\n    if a.shape != b.shape or a.ndim != 2:\n        raise ValueError("metrics require two equally-sized grayscale images")\n    if min(a.shape) < 7:\n        raise ValueError("SSIM requires dimensions >= 7")\n    ac, bc = a - a.mean(), b - b.mean()\n    denominator = float(np.linalg.norm(ac) * np.linalg.norm(bc))\n    correlation = float(np.clip(np.sum(ac * bc) / denominator, -1, 1)) if denominator > 1e-14 else None\n    return {"pearson": correlation, "ssim": float(structural_similarity(a, b, data_range=1.0, win_size=7))}\n\n\ndef phase_metrics(optics, phase, targets):\n    """Evaluate after phase quantization and amplitude PNG quantization."""\n    encoded = encode_phase(phase.detach().cpu().numpy())\n    decoded = torch.as_tensor(decode_phase(encoded), device=phase.device, dtype=phase.dtype)\n    with torch.no_grad():\n        amplitudes = np.abs(optics.physical(decoded).cpu().numpy())\n    source = np.rint(targets.detach().cpu().numpy() * 255).astype(np.uint8)\n    records = []\n    for b in range(len(source)):\n        planes = [metrics(source[b, i], amplitude_png(amplitudes[b, i])[0]) for i in range(4)]\n        correlations = [v["pearson"] for v in planes if v["pearson"] is not None]\n        records.append({"pearson": float(np.mean(correlations)) if correlations else None,\n                        "ssim": float(np.mean([v["ssim"] for v in planes])), "planes": planes})\n    return records\n\n\ndef export_result(output, optics, phase, image_paths, metadata=None):\n    output = Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(f"Refusing to overwrite existing result: {output}")\n    targets = load_quartet(image_paths)\n    if phase.ndim != 3 or phase.shape[0] != 1:\n        raise ValueError("export expects one phase map [1,128,128]")\n    output.mkdir(parents=True, exist_ok=True)\n    encoded = encode_phase(phase[0].detach().cpu().numpy())\n    phase_path = output / "doe_phase.png"\n    Image.fromarray(encoded).save(phase_path)\n    # Intentionally read the saved file back: all reported reconstructions use its quantized phase.\n    with Image.open(phase_path) as saved:\n        decoded = decode_phase(np.array(saved))\n    decoded_tensor = torch.as_tensor(decoded, device=phase.device, dtype=phase.dtype)[None]\n    with torch.no_grad():\n        physical = optics.physical(decoded_tensor)[0].cpu().numpy()\n    amplitudes = np.abs(physical)\n    np.save(output / "phase_rad.npy", decoded)\n    np.save(output / "reconstructed_amplitudes.npy", amplitudes)\n    np.save(output / "reconstructed_complex_fields.npy", physical)\n    input_arrays = np.rint(targets[0].numpy() * 255).astype(np.uint8)\n    rows = []\n    images = []\n    for i, z in enumerate(optics.config.distances_m):\n        image, peak = amplitude_png(amplitudes[i])\n        images.append(image)\n        Image.fromarray(input_arrays[i]).save(output / f"target_{i + 1}.png")\n        Image.fromarray(image).save(output / f"reconstruction_{i + 1}.png")\n        rows.append({"plane": i + 1, "distance_mm": z * 1000,\n                     "output_pixel_pitch_um": optics.config.output_pixel_pitches_m[i] * 1e6,\n                     "amplitude_peak_before_display_scaling": peak,\n                     **metrics(input_arrays[i], image)})\n    report = {\n        "optics": optics.config.to_dict(), "metadata": metadata or {}, "planes": rows,\n        "mean_pearson": float(np.mean([row["pearson"] for row in rows if row["pearson"] is not None]))\n                        if any(row["pearson"] is not None for row in rows) else None,\n        "mean_ssim": float(np.mean([row["ssim"] for row in rows])),\n        "phase_encoding": "uint8 0 -> -pi; 255 -> +pi; round to nearest",\n        "reconstruction": "single-FFT Fresnel, reconstructed from the saved quantized DOE PNG",\n        "png_scaling": "amplitude (not intensity); each plane divided by its own maximum; no background subtraction or fitted gain",\n        "ssim": {"implementation": "skimage.metrics.structural_similarity", "win_size": 7, "data_range": 1.0},\n        "pearson_constant_image": "null (undefined), never a fabricated zero or one",\n        "doe_sha256": hashlib.sha256(phase_path.read_bytes()).hexdigest(),\n        "inputs": [{"name": Path(path).name, "sha256": hashlib.sha256(Path(path).read_bytes()).hexdigest()} for path in image_paths],\n    }\n    (output / "metrics.json").write_text(json.dumps(report, indent=2, allow_nan=False) + "\\n")\n    with (output / "metrics.csv").open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    plot_result(output / "comparison.png", input_arrays, np.stack(images), encoded, rows)\n    return report\n\n\ndef plot_result(path, targets, reconstructions, phase, rows):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    fig, axes = plt.subplots(4, 3, figsize=(10, 11), constrained_layout=True)\n    for i, row in enumerate(rows):\n        axes[i, 0].imshow(targets[i], cmap="gray", vmin=0, vmax=255)\n        axes[i, 0].set_title(f"Target {i+1}, z={row[\'distance_mm\']:.1f} mm")\n        axes[i, 1].imshow(reconstructions[i], cmap="gray", vmin=0, vmax=255)\n        corr = row["pearson"]\n        axes[i, 1].set_title(f"Reconstruction |U|; r={corr:.3f}" if corr is not None else "Reconstruction; r undefined")\n        axes[i, 2].imshow(np.abs(targets[i].astype(float) - reconstructions[i]), cmap="magma", vmin=0, vmax=255)\n        axes[i, 2].set_title(f"Absolute difference; SSIM={row[\'ssim\']:.3f}")\n        for ax in axes[i]:\n            ax.axis("off")\n    fig.savefig(path, dpi=140)\n    plt.close(fig)\n    fig, ax = plt.subplots(figsize=(4, 4), constrained_layout=True)\n    display = ax.imshow(decode_phase(phase), cmap="twilight", vmin=-math.pi, vmax=math.pi)\n    ax.set_title("Quantized DOE phase (radians)")\n    fig.colorbar(display, ax=ax)\n    fig.savefig(Path(path).with_name("phase_preview.png"), dpi=140)\n    plt.close(fig)\n', 'tests/test_deliverables.py': 'import json\nimport math\n\nimport numpy as np\nfrom PIL import Image\nimport pytest\nimport torch\n\nfrom doe.data import generate_dataset, load_quartet, QuartetDataset\nfrom doe.model import FresnelNet\nfrom doe.optics import Fresnel\nfrom doe.results import amplitude_png, decode_phase, encode_phase, export_result, metrics\n\n\ndef test_phase_endpoints_and_quantization_error():\n    phi = np.linspace(-math.pi, math.pi, 1001)\n    encoded = encode_phase(phi)\n    assert encoded.dtype == np.uint8\n    assert encoded[0] == 0 and encoded[-1] == 255\n    np.testing.assert_allclose(decode_phase(np.array([0, 255], dtype=np.uint8)), [-math.pi, math.pi])\n    assert np.max(np.abs(decode_phase(encoded) - phi)) <= math.pi / 255 + 1e-12\n    with pytest.raises(ValueError):\n        encode_phase(np.array([math.nan]))\n\n\ndef test_pearson_ssim_known_cases():\n    a = np.arange(128 * 128).reshape(128, 128).astype(np.uint8)\n    assert metrics(a, a) == {"pearson": 1.0, "ssim": 1.0}\n    assert metrics(a, 255 - a)["pearson"] == pytest.approx(-1.0)\n    assert metrics(np.zeros_like(a), a)["pearson"] is None\n    assert metrics(np.zeros_like(a), np.zeros_like(a))["ssim"] == 1.0\n\n\ndef test_export_reconstructs_saved_quantized_doe_and_metrics(tmp_path):\n    generate_dataset(tmp_path / "data", train=1, validation=1, test=1)\n    inputs = [tmp_path / "data/test/sample_0000" / f"plane_{i}.png" for i in range(1, 5)]\n    optics = Fresnel()\n    torch.manual_seed(11)\n    phase = torch.rand(1, 128, 128) * 2 * math.pi - math.pi\n    out = tmp_path / "result"\n    report = export_result(out, optics, phase, inputs)\n    with Image.open(out / "doe_phase.png") as image:\n        assert image.mode == "L" and image.size == (128, 128)\n        decoded = decode_phase(np.array(image))\n    expected = np.abs(optics.physical(torch.tensor(decoded, dtype=torch.float32)[None])[0].numpy())\n    np.testing.assert_allclose(np.load(out / "reconstructed_amplitudes.npy"), expected)\n    for i in range(4):\n        with Image.open(out / f"reconstruction_{i + 1}.png") as image:\n            assert image.mode == "L" and image.size == (128, 128)\n            png = np.array(image)\n        np.testing.assert_array_equal(png, amplitude_png(expected[i])[0])\n        target = np.array(Image.open(inputs[i]))\n        assert report["planes"][i]["ssim"] == metrics(target, png)["ssim"]\n    assert len(report["planes"]) == 4\n    json.loads((out / "metrics.json").read_text())\n    with pytest.raises(FileExistsError):\n        export_result(out, optics, phase, inputs)\n\n\ndef test_strict_inputs_and_separate_splits(tmp_path):\n    manifest = generate_dataset(tmp_path / "data", train=2, validation=2, test=2)\n    seeds = [s for split in manifest["splits"].values() for row in split for s in row["image_seeds"]]\n    assert len(seeds) == len(set(seeds))\n    assert QuartetDataset(tmp_path / "data", "train")[0].shape == (4, 128, 128)\n    paths = [tmp_path / "data/test/sample_0000" / f"plane_{i}.png" for i in range(1, 5)]\n    Image.new("RGB", (128, 128)).save(tmp_path / "rgb.png")\n    with pytest.raises(ValueError, match="grayscale"):\n        load_quartet([tmp_path / "rgb.png", *paths[1:]])\n    with pytest.raises(ValueError, match="exactly four"):\n        load_quartet(paths[:3])\n    with pytest.raises(FileExistsError):\n        generate_dataset(tmp_path / "data", train=1, validation=1, test=1)\n', 'tests/test_optics.py': '"""Independent quadrature and gradient checks, beyond testing FFT against itself."""\nimport math\n\nimport numpy as np\nimport pytest\nimport torch\n\nfrom doe.model import FresnelNet, physics_loss\nfrom doe.optics import Fresnel, OpticsConfig\n\n\ndef test_fresnel_matches_direct_diffraction_integral():\n    config = OpticsConfig(size=8)\n    optics = Fresnel(config, dtype=torch.float64)\n    rng = np.random.default_rng(17)\n    phase = rng.uniform(-math.pi, math.pi, (8, 8))\n    calculated = optics.physical(torch.tensor(phase)[None])[0].numpy()\n    axis = (np.arange(8) - 4) * config.pixel_pitch_m\n    u, v = np.meshgrid(axis, axis)\n    for i, z in enumerate(config.distances_m):\n        out_axis = (np.arange(8) - 4) * config.output_pixel_pitches_m[i]\n        x, y = np.meshgrid(out_axis, out_axis)\n        distance_squared = (x[:, :, None, None] - u)**2 + (y[:, :, None, None] - v)**2\n        integrand = np.exp(1j * phase) * np.exp(1j * math.pi * distance_squared / (config.wavelength_m * z))\n        direct = (integrand.sum(axis=(-2, -1)) * config.pixel_pitch_m**2\n                  * np.exp(2j * math.pi * z / config.wavelength_m) / (1j * config.wavelength_m * z))\n        np.testing.assert_allclose(calculated[i], direct, rtol=1e-8, atol=1e-8)\n\n\ndef test_adjoint_and_physical_power_with_different_output_pixel_sizes():\n    torch.manual_seed(7)\n    config = OpticsConfig(size=16)\n    optics = Fresnel(config, dtype=torch.float64)\n    field = torch.randn(2, 16, 16, dtype=torch.complex128)\n    propagated = optics.field(field)\n    arbitrary = torch.randn_like(propagated)\n    lhs = (propagated.conj() * arbitrary).sum()\n    rhs = (field[:, None].conj() * optics.adjoint(arbitrary)).sum()\n    torch.testing.assert_close(lhs, rhs)\n    phase = torch.rand(2, 16, 16, dtype=torch.float64)\n    power = optics.physical(phase).abs().square().sum(dim=(-2, -1))\n    pitches = torch.tensor(config.output_pixel_pitches_m, dtype=torch.float64)\n    np.testing.assert_allclose((power * pitches.square()).numpy(), config.size**2 * config.pixel_pitch_m**2, rtol=1e-9)\n\n\ndef test_phase_gradient_matches_finite_difference():\n    torch.manual_seed(20)\n    optics = Fresnel(OpticsConfig(size=8), dtype=torch.float64)\n    phase = torch.randn(1, 8, 8, dtype=torch.float64, requires_grad=True)\n    targets = torch.rand(1, 4, 8, 8, dtype=torch.float64) + 0.1\n    physics_loss(optics, phase, targets).backward()\n    index = (0, 2, 3)\n    step = 1e-5\n    plus, minus = phase.detach().clone(), phase.detach().clone()\n    plus[index] += step\n    minus[index] -= step\n    numerical = (physics_loss(optics, plus, targets) - physics_loss(optics, minus, targets)) / (2 * step)\n    torch.testing.assert_close(phase.grad[index], numerical, rtol=1e-5, atol=1e-7)\n\n\ndef test_neural_weights_receive_finite_nonzero_gradients():\n    torch.manual_seed(5)\n    model = FresnelNet(OpticsConfig(size=16), stages=2, width=4)\n    targets = torch.rand(2, 4, 16, 16) + 0.1\n    loss = physics_loss(model.optics, model(targets), targets)\n    loss.backward()\n    gradient = model.corrector[-1].weight.grad\n    assert torch.isfinite(gradient).all()\n    assert gradient.abs().max() > 0\n    assert torch.isfinite(model.plane_logits.grad).all()\n    with pytest.raises(ValueError, match="black"):\n        model(torch.zeros_like(targets))\n', 'scripts/build_notebook.py': '"""Build a standalone, inspectable Kaggle notebook with embedded project source."""\nimport json\nfrom pathlib import Path\n\nimport nbformat as nbf\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef build():\n    files = [*sorted((ROOT / "doe").glob("*.py")), *sorted((ROOT / "tests").glob("*.py")),\n             *sorted((ROOT / "scripts").glob("*.py")), *sorted((ROOT / "docs").glob("*.md")),\n             ROOT / "pyproject.toml", ROOT / "requirements.txt", ROOT / "requirements-dev.txt", ROOT / "README.md"]\n    sources = {str(path.relative_to(ROOT)): path.read_text() for path in files}\n    cells = [nbf.v4.new_markdown_cell(r"""# Нейросетевой синтез четырёхплоскостного ДОЭ\n\nЗадание хакатона: 4 одноканальных PNG 128×128 → фазовый ДОЭ 128×128 →\n4 реконструкции методом Френеля; λ=532 нм, пиксель ПВМС 10 мкм,\nz=[24.1,25.3,26.5,27.7] мм. Экспорт: 8-битные PNG, Pearson и SSIM.\n\n**Run All** запускает обучение, тестовую оценку и экспорт. Включите\nSettings → Accelerator → GPU T4 ×2 при доступной квоте; код также работает на CPU.\nИнтернет и Kaggle API-токен внутри notebook не требуются.\n\nВ документах нет четырёх исходных объектных PNG: по умолчанию используется явно\nобозначенный процедурный демонстрационный набор. Для своих данных укажите четыре\nпути в `INPUT_IMAGES` ниже. Их порядок соответствует возрастающему z.\n\nМетод — фиксированная развёрнутая нейросеть с оптическими проекциями и обучаемыми\nCNN-коррекциями. Обучение без эталонных голограмм; при инференсе нет шагов оптимизатора.\nТестовые изображения не участвуют в обучении и выборе checkpoint.\n"""), nbf.v4.new_code_cell(\n        "# Embedded source: inspectable, no Git clone or external download.\\n"\n        "from pathlib import Path\\nimport json, os, sys\\n"\n        "BASE = Path(\'/kaggle/working\') if Path(\'/kaggle/working\').exists() else Path.cwd() / \'notebook-workdir\'\\n"\n        "PROJECT = BASE / \'nuclear-hack-source\'\\nPROJECT.mkdir(parents=True, exist_ok=True)\\n"\n        + "SOURCE_FILES = " + repr(sources) + "\\n"\n        "for name, source in SOURCE_FILES.items():\\n"\n        "    path = PROJECT / name\\n"\n        "    path.parent.mkdir(parents=True, exist_ok=True)\\n"\n        "    if path.exists() and path.read_text() != source:\\n"\n        "        raise RuntimeError(f\'Existing source differs: {path}; preserve it and use a fresh work directory\')\\n"\n        "    path.write_text(source)\\n"\n        "sys.path.insert(0, str(PROJECT))\\n"\n        "import torch, numpy as np, PIL, skimage, matplotlib\\n"\n        "print({\'torch\': torch.__version__, \'cuda\': torch.cuda.is_available(), \'numpy\': np.__version__})\\n"\n        "if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))\\n"\n    ), nbf.v4.new_markdown_cell("""## Настройки\n\nЧетыре входных файла должны быть PNG, режим L, 8 бит, 128×128. Пустая плоскость\nнесовместима с фазовым ДОЭ при равномерном освещении и не принимается.\n`INPUT_IMAGES=[]` означает демонстрацию, а не выполнение на официальных данных.\nCPU использует меньший учебный набор; на GPU можно увеличить число эпох и примеров.\nПри повторном запуске создаётся новый каталог, предыдущие результаты сохраняются.\n"""), nbf.v4.new_code_cell("""from datetime import datetime, timezone\nfrom doe.engine import TrainConfig, train, evaluate, predict\nfrom doe.data import generate_dataset\n\nINPUT_IMAGES = []  # [\'/kaggle/input/YOUR_DATA/image1.png\', ..., \'image4.png\']\nQUICK_CHECK = os.environ.get(\'DOE_QUICK_CHECK\') == \'1\'  # integration validation only\nGPU = torch.cuda.is_available()\nRUN = BASE / (\'doe-run-\' + datetime.now(timezone.utc).strftime(\'%Y%m%d-%H%M%S-%f\'))\nRUN.mkdir(parents=True)\nDATA = RUN / \'data\'\nTRAINING = RUN / \'training\'\nif QUICK_CHECK:\n    counts = (8, 4, 4)\n    config = TrainConfig(epochs=2, batch_size=2, stages=2, width=4)\nelse:\n    counts = (512, 64, 32) if GPU else (96, 16, 16)\n    config = TrainConfig(epochs=20 if GPU else 24, batch_size=8 if GPU else 4,\n                         stages=8 if GPU else 6, width=16 if GPU else 12)\nprint(\'Run:\', RUN, \'\\\\nTraining:\', config)\n"""), nbf.v4.new_markdown_cell("""## Данные и обучение\n\nРазные seed для train/validation/test; сеть минимизирует ошибку амплитуд после\nдифференцируемого одно-FFT оператора Френеля. Лучшая модель выбирается по validation loss.\n`initial.pt` нужен для контроля того, чему научилась сеть. `best.pt` — для инференса.\n"""), nbf.v4.new_code_cell("""generate_dataset(DATA, train=counts[0], validation=counts[1], test=counts[2])\ntraining_report = train(DATA, TRAINING, config)\nassert training_report[\'parameter_l2_change_last_vs_initial\'] > 0\nprint(\'Best validation epoch:\', training_report[\'best_epoch\'])\n"""), nbf.v4.new_markdown_cell("""## Независимая тестовая оценка\n\nСравнение с той же сетью до обучения и многоплоскостным Герчбергом–Сэкстоном\nпри одинаковом и большем числе оптических шагов. Все методы оцениваются после\nквантования фазы и амплитуд в 8 бит. Время отражает batched throughput на текущем\nустройстве; ускорение относительно классического метода не предполагается заранее.\n"""), nbf.v4.new_code_cell("""benchmark = evaluate(DATA, TRAINING / \'best.pt\', RUN / \'benchmark.json\',\n                     batch_size=config.batch_size, gs_steps=12 if QUICK_CHECK else 80)\nprint(json.dumps(benchmark[\'summary\'], indent=2))\n"""), nbf.v4.new_markdown_cell("""## ДОЭ и восстановленные изображения\n\nСеть выдаёт фазу, кодируемую как `0 → −π`, `255 → +π` с округлением.\nРеконструкции рассчитываются заново **из сохранённого PNG ДОЭ**. Сохраняется модуль\nкомплексного поля, не квадрат модуля. Для отображения каждый модуль делится на свой\nмаксимум; без вычитания фона и без подгонки к целевому изображению.\nМетрики считаются по фактическим сохранённым 8-битным PNG. В `.npy` сохранены также\nмодули до масштабирования и комплексные поля. Шаг выходной сетки зависит от z:\nΔx_out = λz/(NΔu); изображения разных плоскостей задаются на соответствующих сетках.\n"""), nbf.v4.new_code_cell("""if INPUT_IMAGES:\n    if len(INPUT_IMAGES) != 4:\n        raise ValueError(\'Specify exactly four paths in increasing distance order\')\n    input_paths = [Path(p) for p in INPUT_IMAGES]\n    data_source = \'user-provided quartet\'\nelse:\n    input_paths = [DATA / \'test/sample_0000\' / f\'plane_{i}.png\' for i in range(1, 5)]\n    data_source = \'synthetic held-out demonstration, not official hackathon inputs\'\nresult = predict(TRAINING / \'best.pt\', input_paths, RUN / \'result\', source=data_source)\nprint(json.dumps(result[\'planes\'], indent=2))\nfrom IPython.display import display, Image as DisplayImage\ndisplay(DisplayImage(filename=str(RUN / \'result/comparison.png\')))\ndisplay(DisplayImage(filename=str(RUN / \'result/phase_preview.png\')))\n"""), nbf.v4.new_markdown_cell("""## Материалы для экспертов\n\nАрхив содержит код, полный сгенерированный набор, веса, историю обучения,\nтестовое сравнение, входные PNG, фазовый ДОЭ, четыре восстановленных PNG,\nметрики и исходные комплексные поля. Качество на официальной четвёрке необходимо\nизмерить отдельно после её предоставления. Это численное моделирование; оптическая\nпроверка на ПВМС здесь не выполняется.\n"""), nbf.v4.new_code_cell("""import shutil\nsubmission = RUN / \'submission\'\nsubmission.mkdir()\nfor name, source in SOURCE_FILES.items():\n    path = submission / \'source\' / name\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(source)\nfor name in (\'data\', \'training\', \'result\'):\n    shutil.copytree(RUN / name, submission / name)\nshutil.copy2(RUN / \'benchmark.json\', submission / \'benchmark.json\')\n(submission / \'runtime.json\').write_text(json.dumps({\'torch\': torch.__version__, \'numpy\': np.__version__,\n    \'pillow\': PIL.__version__, \'skimage\': skimage.__version__, \'device\': benchmark[\'device\'],\n    \'input_source\': data_source, \'quick_check\': QUICK_CHECK}, indent=2))\narchive = shutil.make_archive(str(RUN / \'submission\'), \'zip\', submission)\nprint(\'Submission archive:\', archive)\nprint(\'Saved results:\', RUN / \'result\')\n""")]\n    for index, cell in enumerate(cells):\n        cell.id = f"section-{index:02d}"\n    notebook = nbf.v4.new_notebook(cells=cells, metadata={\n        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},\n        "language_info": {"name": "python"},\n        "kaggle": {"accelerator": "gpu", "isInternetEnabled": False},\n    })\n    nbf.validate(notebook)\n    out = ROOT / "notebooks/fresnel_doe_kaggle.ipynb"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    nbf.write(notebook, out)\n    metadata = {"id": "fiskarik/nuclear-hack-fresnel-doe", "title": "Nuclear Hack Fresnel DOE",\n                "code_file": out.name, "language": "python", "kernel_type": "notebook",\n                "is_private": True, "enable_gpu": True, "enable_internet": False,\n                "dataset_sources": [], "competition_sources": [], "kernel_sources": [], "model_sources": []}\n    (out.parent / "kernel-metadata.json").write_text(json.dumps(metadata, indent=2) + "\\n")\n    print(out)\n\n\nif __name__ == "__main__":\n    build()\n', 'scripts/package_submission.py': '"""Package only declared project/data/run paths, never credentials or Git metadata."""\nimport argparse\nfrom pathlib import Path\nimport zipfile\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef package(data, run, output, holdout=None):\n    data, run, output = Path(data).resolve(), Path(run).resolve(), Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(f"Refusing to overwrite {output}")\n    required = [data / "manifest.json", run / "best.pt", run / "training.json", run / "benchmark.json",\n                run / "result/doe_phase.png", run / "result/metrics.json", ROOT / "notebooks/fresnel_doe_kaggle.ipynb"]\n    if holdout:\n        required.append(Path(holdout) / "manifest.json")\n    for path in required:\n        if not path.is_file():\n            raise FileNotFoundError(path)\n    entries = []\n    for name in ("README.md", "pyproject.toml", "requirements.txt", "requirements-dev.txt", ".gitignore"):\n        entries.append((ROOT / name, "source/" + name))\n    for folder in ("doe", "tests", "scripts", "docs", "notebooks"):\n        for path in sorted((ROOT / folder).rglob("*")):\n            if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".pyc":\n                entries.append((path, "source/" + str(path.relative_to(ROOT))))\n    for prefix, directory in (("data", data), ("run", run), ("independent_holdout", Path(holdout).resolve() if holdout else None)):\n        if directory is None:\n            continue\n        for path in sorted(directory.rglob("*")):\n            if path.is_file() and path != output:\n                entries.append((path, prefix + "/" + str(path.relative_to(directory))))\n    if len({name for _, name in entries}) != len(entries):\n        raise ValueError("duplicate archive paths")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(output, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path, name in entries:\n            archive.write(path, name)\n    with zipfile.ZipFile(output) as archive:\n        corrupt = archive.testzip()\n        if corrupt:\n            raise RuntimeError(f"Archive CRC failure: {corrupt}")\n    print(f"{output}: {len(entries)} files, {output.stat().st_size:,} bytes")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data", required=True)\n    parser.add_argument("--run", required=True)\n    parser.add_argument("--holdout")\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args()\n    package(**vars(args))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/validate_notebook.py': '"""Execute every notebook cell; quick mode validates integration, not model quality."""\nimport argparse\nimport os\nfrom pathlib import Path\n\nimport nbformat\nfrom nbclient import NotebookClient\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--notebook", default="notebooks/fresnel_doe_kaggle.ipynb")\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--workdir", required=True)\n    parser.add_argument("--kernel", default="python3")\n    parser.add_argument("--quick-check", action="store_true")\n    args = parser.parse_args()\n    notebook = nbformat.read(args.notebook, as_version=4)\n    workdir = Path(args.workdir).resolve()\n    workdir.mkdir(parents=True, exist_ok=True)\n    output = Path(args.output)\n    if output.exists():\n        raise FileExistsError(output)\n    if args.quick_check:\n        os.environ["DOE_QUICK_CHECK"] = "1"\n    else:\n        os.environ.pop("DOE_QUICK_CHECK", None)\n    client = NotebookClient(notebook, timeout=1800, kernel_name=args.kernel,\n                            resources={"metadata": {"path": str(workdir)}})\n    client.execute()\n    executed = 0\n    for cell in notebook.cells:\n        if cell.cell_type == "code":\n            assert cell.execution_count is not None\n            assert not any(x.output_type == "error" for x in cell.outputs)\n            executed += 1\n    notebook.metadata["validation"] = {"quick_check": args.quick_check, "executed_code_cells": executed}\n    output.parent.mkdir(parents=True, exist_ok=True)\n    nbformat.write(notebook, output)\n    print(f"Executed {executed} code cells; quick_check={args.quick_check}; {output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'docs/validation.md': '# Фактическая проверка решения\n\nПроверено в текущей облачной среде на CPU. GPU и удалённое выполнение в Kaggle\n**не проверены**: в текущей среде нет Kaggle API-токена и применённого сетевого\nразрешения для Kaggle. Notebook можно импортировать и выполнить вручную в Kaggle\nбез передачи токена в код.\n\n## Обучение\n\n- Python 3.12.14, PyTorch 2.9.1+cpu, NumPy 2.3.5; полный состав —\n  `docs/cpu-environment.txt`.\n- Процедурный набор `data/procedural_v1`: 96/16 изображений-четвёрок train/validation;\n  seed 2026. Каждая четвёрка содержит четыре PNG/L/128×128.\n- 6 оптических слоёв, CNN width=12, batch=4, Adam lr=0.005, 24 эпохи, seed=42,\n  2 CPU-потока, cosine scheduler.\n- Validation loss до обучения: **0.9402181**; лучший: **0.9188025**, эпоха 24.\n- Обучение заняло около **140 секунд** в этом запуске. Веса действительно\n  обновлены; величина изменения параметров и полная история есть в `training.json`.\n- Сеть с обучаемым over-relaxation выбрана по validation. Ранний вариант\n  использовался для разработки; его тестовые результаты не выдаются за финальную\n  независимую оценку. Данные ниже — новая выборка с другим seed.\n\n## Независимая финальная выборка\n\n`data/independent_holdout`, seed **91007**, 32 новых четвёрки. Эта выборка не\nиспользовалась для обучения и выбора checkpoint. В manifest сохранены seed и\nSHA256; в `benchmark.json` — результаты каждой четвёрки. Показатели усреднены\nпо четырём плоскостям и затем по 32 четвёркам.\n\n| Метод | Pearson ↑ | SSIM ↑ | мс/четвёрку ↓ |\n|---|---:|---:|---:|\n| FresnelNet, обученная | 0.59643 | 0.09554 | 43.03 |\n| Та же сеть до обучения | 0.58689 | 0.09028 | 44.61 |\n| GS, 6 шагов | 0.58689 | 0.09028 | 25.94 |\n| GS, 80 шагов | 0.61833 | 0.12444 | 158.99 |\n\nИзмерение времени — один проход после прогрева, batch=4; это batched throughput,\nне latency одиночного запроса и не статистическая оценка производительности.\nЗагрузка PNG, обучение, расчёт метрик и запись результатов исключены из времени\nсинтеза. Округление phase/amplitude в 8 бит включено в качество всех методов.\n\nОбучение даёт умеренное улучшение относительно такой же сети до обучения и GS\nс тем же числом шагов. GS за 80 шагов качественнее, но медленнее в данном CPU-прогоне.\n**Преимущество по качеству над классическим методом не заявляется.** Малый SSIM\nи видимые помехи означают, что это рабочая исходная реализация, а не решение с\nгарантированно высоким качеством. Для соревнования нужны проверка на реальных\nизображениях, расширение обучающего набора и дальнейшая настройка.\n\n## Экспортированная демонстрация\n\nИспользована первая четвёрка независимой выборки, `test/sample_0000`, а не отобранный\nлучший пример. Исходные PNG, ДОЭ и реконструкции находятся в `run/result` архива.\n\n| z, мм | Pearson | SSIM |\n|---:|---:|---:|\n| 24.1 | 0.68916 | 0.14880 |\n| 25.3 | 0.39648 | 0.08306 |\n| 26.5 | 0.62391 | 0.07223 |\n| 27.7 | 0.65105 | 0.07878 |\n\nСреднее: Pearson **0.59015**, SSIM **0.09572**. PNG амплитуд масштабированы по\nсобственному максимуму; gain по целевым изображениям не подбирался. Видимые спекл\nи межплоскостные помехи сохранены, фильтрация не применялась.\n\n## Что прошло\n\n- **8 тестов pytest**, 0 пропущенных: прямая квадратура дифракции без FFT,\n  сопряжённость, сохранение физической мощности на разных сетках, finite-difference\n  градиент фазы, градиенты CNN, кодирование фазы, Pearson/SSIM, строгие PNG,\n  воспроизводимые данные, реконструкции сохранённого квантованного ДОЭ.\n- Все **6 исполняемых ячеек** автономного notebook выполнены локально через Jupyter.\n  Для этой интеграционной проверки включён `DOE_QUICK_CHECK=1`: 8/4/4 примера,\n  2 эпохи, 2 слоя, width=4. Проверены обучение, benchmark, экспорт изображений и\n  сборка ZIP. Это **не** проверка полной GPU-конфигурации notebook в Kaggle.\n- Полное обучение основной CPU-модели и оценка 32 новых четвёрок выполнены отдельно\n  от короткого notebook-прогона.\n\n## Что остаётся\n\nВ исходных DOCX/PDF нет четырёх PNG объектов организаторов: результаты выше\nотносятся только к процедурной демонстрации. Для выполнения на нужной четвёрке\nпередайте её в `doe predict` или `INPUT_IMAGES` notebook. Код не подменяет и не\nдорисовывает официальные входы.\n\nДля автоматического запуска в аккаунте `fiskarik` требуются защищённый\n`KAGGLE_API_TOKEN` и разрешённый сетевой доступ. Настройки сохранены в черновике\nсреды ранее; их сохранение и публикация не подтверждают выполнение в Kaggle.\nЭкспериментальная проверка оптики на физическом ПВМС не проводилась.\n', 'pyproject.toml': '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "fresnel-doe"\nversion = "0.1.0"\ndescription = "Physics-informed neural synthesis of a four-plane phase-only diffractive optical element"\nrequires-python = ">=3.11"\ndependencies = [\n  "torch>=2.4,<3",\n  "numpy>=1.26,<3",\n  "Pillow>=10.1,<13",\n  "scikit-image>=0.24,<1",\n  "matplotlib>=3.8,<4",\n]\n\n[project.optional-dependencies]\ndev = ["pytest>=8,<9", "nbformat>=5.10,<6", "nbclient>=0.10,<1", "ipykernel>=6.29,<7"]\n\n[project.scripts]\nfresnel-doe = "doe.cli:main"\n\n[tool.setuptools.packages.find]\ninclude = ["doe*"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n', 'requirements.txt': '# PyTorch is already installed on Kaggle; retain its CUDA build there.\n# For local CPU setup, install torch from https://download.pytorch.org/whl/cpu first.\ntorch==2.9.1\nnumpy==2.3.5\nPillow==12.0.0\nscikit-image==0.25.2\nmatplotlib==3.10.7\n', 'requirements-dev.txt': 'pytest==8.4.2\nnbformat==5.10.4\nnbclient==0.10.2\nipykernel==6.30.1\n', 'README.md': "# Нейросетевой синтез четырёхплоскостного ДОЭ\n\nРешение задания хакатона из «Хакатон_задача_q2.docx» и «AI фотоника.pdf»:\nчетыре изображения → один фазовый дифракционный оптический элемент → четыре\nчисленно восстановленных изображения и метрики качества.\n\n**В документах нет исходной четвёрки объектных PNG.** В репозитории есть\nпроцедурный демонстрационный набор, обучение и проверка на независимом test split.\nКачество на изображениях организаторов нужно измерить отдельно. Симуляция не\nзаменяет эксперимент на физическом ПВМС.\n\n## Готовые материалы\n\n- [Полный архив для экспертов](deliverables/submission.zip): исходники, данные,\n  checkpoint, обучение, независимая проверка и изображения.\n- [Обученные веса](deliverables/best.pt) — готовая CPU-модель, 6 слоёв, 24 эпохи.\n- [Пример ДОЭ и четырёх реконструкций](deliverables/result/).\n- [Результаты независимой проверки](deliverables/benchmark.json) и\n  [история обучения](deliverables/training.json).\n- [Kaggle Notebook](notebooks/fresnel_doe_kaggle.ipynb).\n\nДля инференса без повторного обучения используйте `--checkpoint deliverables/best.pt`\nв команде `doe predict`. После распаковки полного архива этот checkpoint также\nнаходится в `run/best.pt`. Веса загружаются с `weights_only=True`.\n\n## Что реализовано\n\n- `FresnelNet`: обучаемая развёрнутая сеть с физическими слоями и CNN-коррекциями;\n  вход `[B,4,128,128]`, выход `[B,128,128]` в пределах `[-π,π]`.\n- Дифференцируемый прямой расчёт Френеля с **одним FFT на плоскость**.\n- Фазовый ДОЭ `doe_phase.png`: PNG/L/8 бит/128×128, `0 → −π`, `255 → +π`.\n- Четыре `reconstruction_1.png` … `reconstruction_4.png`: модули полей,\n  полученных **из сохранённого, квантованного PNG ДОЭ**.\n- Pearson и SSIM для каждой пары исходное/восстановленное; JSON и CSV.\n- Неокруглённые амплитуды и комплексные поля `.npy`, физические параметры,\n  контрольные суммы входов, ДОЭ и checkpoint.\n- Обучение без размеченных голограмм, train/validation/test, контроль до обучения,\n  сравнение с многоплоскостным алгоритмом Герчберга–Сэкстона.\n- Самодостаточный Kaggle notebook, тесты, веса и материалы для экспертов.\n\n## Запуск в Kaggle\n\nИмпортируйте **[notebooks/fresnel_doe_kaggle.ipynb](notebooks/fresnel_doe_kaggle.ipynb)**\nчерез Kaggle → New Notebook → File → Import Notebook. Включите GPU при доступной\nквоте и выполните Run All. Весь Python-код встроен в notebook обычным текстом;\nклонирование GitHub, интернет и API-токен **внутри Kaggle** не требуются.\nИспользуются предустановленные PyTorch, NumPy, Pillow, scikit-image и matplotlib.\nВерсии фактического окружения сохраняются в `runtime.json`.\n\nNotebook генерирует данные, обучает сеть, оценивает test split, экспортирует\nрезультат и собирает `submission.zip` со всеми материалами. На GPU по умолчанию\n512/64/32 четвёрки train/validation/test, 20 эпох, 8 оптических слоёв. На CPU —\n96/16/16, 24 эпохи, 6 слоёв. При наличии своих PNG добавьте их как Kaggle Dataset и\nзадайте **четыре конкретных пути** в `INPUT_IMAGES` в порядке расстояний ниже.\nДемонстрационный режим явно отмечается в выходном отчёте.\n\nДля отправки из облачной среды через CLI нужен её настроенный доступ к Kaggle:\n\n```bash\n/workspace/.tools/kaggle/bin/kaggle kernels push -p notebooks --accelerator NvidiaTeslaT4\n/workspace/.tools/kaggle/bin/kaggle kernels status fiskarik/nuclear-hack-fresnel-doe\n/workspace/.tools/kaggle/bin/kaggle kernels output fiskarik/nuclear-hack-fresnel-doe -p artifacts/kaggle\n```\n\n`push` создаёт/обновляет и запускает приватный notebook; перед первым push убедитесь,\nчто slug не занят другой вашей работой. API-токен вводится только в защищённых\nнастройках облачной среды как `KAGGLE_API_TOKEN`. Статус `complete` и проверка\nрезультатов необходимы, прежде чем считать удалённый запуск успешным. Сетевой доступ,\nверификация аккаунта и доступность ускорителя определяются средой и Kaggle.\n\n## Локальный запуск\n\nPython 3.11+, рекомендуется 3.12. Для CPU можно установить PyTorch отдельно:\n\n```bash\npython3 -m venv .venv\nsource .venv/bin/activate\npython -m pip install torch==2.9.1 --index-url https://download.pytorch.org/whl/cpu\npython -m pip install -r requirements.txt\npython -m pip install -e '.[dev]'\npython -m pytest -q\n```\n\nНе заменяйте CUDA-сборку PyTorch в Kaggle локальной CPU-сборкой. `requirements.txt`\nфиксирует основной проверенный стек; полный состав CPU-окружения записан в\n`docs/cpu-environment.txt`. В подготовленной облачной среде интерпретатор находится\nв `/workspace/.tools/photonics/bin/python`; команды выполняются из `/workspace/Nuclear-hack`.\n\n```bash\npython -m doe generate --output data/my-run --train 96 --validation 16 --test 16\npython -m doe train --data data/my-run --output artifacts/my-run \\\n  --epochs 24 --learning-rate 0.005 --stages 6 --width 12 --batch-size 4 --device auto\npython -m doe evaluate --data data/my-run --checkpoint artifacts/my-run/best.pt \\\n  --output artifacts/my-run/benchmark.json\npython -m doe predict --checkpoint artifacts/my-run/best.pt \\\n  --images image1.png image2.png image3.png image4.png --output artifacts/result\n```\n\nРезультаты предыдущего запуска не перезаписываются: используйте новый каталог.\nИзображения должны быть именно PNG/L/8 бит/128×128; код не меняет размер и не\nпревращает RGB в grayscale молча. Порядок файлов — `24.1, 25.3, 26.5, 27.7 мм`.\nПолностью чёрная плоскость отклоняется: фазовый ДОЭ при равномерном ненулевом\nосвещении сохраняет энергию и не может дать нулевое поле на всей выходной плоскости.\n\n## Физическая модель\n\n| Параметр | Значение |\n|---|---|\n| Размер | 128×128 |\n| Длина волны λ | 532 нм |\n| Входной шаг Δu=Δv | 10 мкм |\n| Расстояния z | 24.1, 25.3, 26.5, 27.7 мм |\n| Освещение | равномерное, амплитуда 1 |\n| ДОЭ | H(u,v)=exp(iφ(u,v)), φ∈[−π,π] |\n\nИспользуется формула из задания с дискретизационным множителем Δu²:\n\n```text\nU_z(x,y) = exp(ikz)/(i λ z) · exp[iπ(x²+y²)/(λz)]\n           · Δu² · FFT{exp(iφ(u,v)) · exp[iπ(u²+v²)/(λz)]},  k=2π/λ.\n```\n\nНачало координат находится в центре массива. FFT центрированное. При FFT с\n`norm='ortho'` восстанавливающий множитель равен `N Δu²/(iλz) · exp(ikz)`.\nНикакой подмены одно-FFT модели на angular spectrum или свёртку нет.\n\n**Сетка восстановления зависит от расстояния**:\n`Δx_z=Δy_z=λz/(NΔu)`, то есть примерно\n`10.0166, 10.5153, 11.0141, 11.5128 мкм`. Каждый входной объект трактуется на\nсвоей соответствующей выходной сетке. Это не четыре изображения с одним и тем же\nфизическим шагом. В отчёте сохранён шаг для каждой плоскости.\n\nДля устойчивого обучения оператор нормирован до унитарного: удалён только\nоднородный комплексный множитель каждой плоскости. При экспорте полный физический\nмножитель возвращается. Это не меняет форму модуля, Pearson и масштабированных PNG.\n\n## Нейросетевой метод и обучение\n\n1. Четыре входные амплитуды нормируются до единичного RMS: энергия равномерного\n   фазового ДОЭ фиксирована. Градации серого интерпретируются как **амплитуда**, как\n   требуется при сравнении с модулями в задании, а не как интенсивность.\n2. Детерминированные случайные фазы в плоскостях объектов и обратные операторы\n   Френеля дают начальное поле на ДОЭ. В тестовом режиме используются те же правила.\n3. Каждый слой сети вычисляет четыре поля, накладывает ограничение на амплитуду,\n   переносит их обратно и объединяет с обучаемыми весами. Обучаемое смешивание\n   и общая небольшая CNN корректируют фазу. Модуль поля на ДОЭ остаётся единичным.\n4. Число слоёв фиксировано. При инференсе выполняется один проход сети;\n   **шагов Adam, поиска лучшей фазы и дообучения на тестовом изображении нет**.\n   Каждый слой содержит прямые и обратные FFT: это физически информированная\n   развёрнутая сеть, а не обещание «синтеза без дифракционных расчётов».\n5. Loss состоит из MSE амплитуд с единичным RMS, многомасштабной ошибки и ошибки\n   корреляции. Используются Adam, ограничение градиента и cosine learning rate.\n   Голограммы-эталоны для обучения не требуются.\n\nПроверочные данные — цифры, диски, прямоугольники, кольца и линии с варьируемой\nгеометрией и сглаживанием. Генератор и seed входят в поставку; PNG, manifest и\nSHA256 позволяют воспроизвести набор. Три split имеют разные seed. Checkpoint\nвыбирается **только** по validation loss; отдельный test split используется для\nфинальной оценки. Обучение не гарантирует идеального восстановления четырёх\nнезависимых объектов одним фазовым экраном: остаются спекл, межплоскостные помехи\nи ограничения числа степеней свободы.\n\n## Метрики и честное сравнение\n\n- Фаза сначала округляется в 8 бит, PNG читается обратно. Реконструкции считаются\n  из этой фазы — потеря качества квантования включена в оценку.\n- Сохраняется `|U|`, не `|U|²`. Для PNG каждая плоскость делится на собственный\n  максимум и округляется до 8 бит. Нет подгонки gain по target, вычитания фона,\n  фильтрации или изменения размера. Исходные физические модули сохранены в `.npy`.\n- Pearson вычисляется по центрированным пикселям; для постоянного изображения\n  он математически не определён и записывается как `null`.\n- SSIM: `skimage.metrics.structural_similarity`, окно 7×7, `data_range=1` после\n  деления фактических PNG на 255; используется стандартная локальная версия SSIM.\n- Сравниваются обученная сеть, такая же сеть до обучения и GS с малым/большим\n  числом шагов. Все используют одинаковую физику, входы и квантование.\n- Время измеряется после прогрева с синхронизацией CUDA и не включает загрузку\n  файлов и метрики. В benchmark указано **batched seconds per quartet**, не latency\n  одиночного запроса. Сбор данных и обучение в время инференса не входят.\n- Превосходство по скорости или качеству не предполагается заранее: результаты\n  конкретного измерения находятся в `benchmark.json` и `docs/validation.md`.\n\n## Проверки и структура\n\n```text\ndoe/optics.py       — физика, центрированные FFT и сопряжённый оператор\ndoe/model.py        — нейросеть, loss, классический GS\ndoe/data.py         — строгий ввод PNG и воспроизводимый набор\ndoe/engine.py       — обучение, выбор checkpoint, оценка, инференс\ndoe/results.py      — квантование, реконструкции, метрики, визуализация\ntests/             — независимая квадратура Френеля, adjoint, мощность,\n                     finite differences, CNN-градиенты, PNG и метрики\nnotebooks/         — автономный notebook и приватные метаданные Kaggle\nscripts/           — генерация notebook и архива для экспертов\ndocs/              — фактический отчёт валидации и состав окружения\n```\n\nОптический оператор сравнивается с прямой суммой интеграла дифракции **без FFT**\nна малой сетке. Проверяются сохранение физической мощности с учётом выходного\nшага, сопряжённость, градиент конечными разностями, границы фазы, строгий формат\nPNG, повторное восстановление сохранённого ДОЭ и метрики известных случаев.\n\nОбновление notebook после изменения кода: `python scripts/build_notebook.py`.\nСборка материалов:\n\n```bash\npython scripts/package_submission.py --data data/procedural_v1 \\\n  --run artifacts/cpu_v2 --holdout data/independent_holdout --output artifacts/submission.zip\n```\n"}
for name, source in SOURCE_FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_text() != source:
        raise RuntimeError(f'Existing source differs: {path}; preserve it and use a fresh work directory')
    path.write_text(source)
sys.path.insert(0, str(PROJECT))
import torch, numpy as np, PIL, skimage, matplotlib
print({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'numpy': np.__version__})
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))


## Настройки

Четыре входных файла должны быть PNG, режим L, 8 бит, 128×128. Пустая плоскость
несовместима с фазовым ДОЭ при равномерном освещении и не принимается.
`INPUT_IMAGES=[]` означает демонстрацию, а не выполнение на официальных данных.
CPU использует меньший учебный набор; на GPU можно увеличить число эпох и примеров.
При повторном запуске создаётся новый каталог, предыдущие результаты сохраняются.


In [ ]:
from datetime import datetime, timezone
from doe.engine import TrainConfig, train, evaluate, predict
from doe.data import generate_dataset

INPUT_IMAGES = []  # ['/kaggle/input/YOUR_DATA/image1.png', ..., 'image4.png']
QUICK_CHECK = os.environ.get('DOE_QUICK_CHECK') == '1'  # integration validation only
GPU = torch.cuda.is_available()
RUN = BASE / ('doe-run-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f'))
RUN.mkdir(parents=True)
DATA = RUN / 'data'
TRAINING = RUN / 'training'
if QUICK_CHECK:
    counts = (8, 4, 4)
    config = TrainConfig(epochs=2, batch_size=2, stages=2, width=4)
else:
    counts = (512, 64, 32) if GPU else (96, 16, 16)
    config = TrainConfig(epochs=20 if GPU else 24, batch_size=8 if GPU else 4,
                         stages=8 if GPU else 6, width=16 if GPU else 12)
print('Run:', RUN, '\nTraining:', config)


## Данные и обучение

Разные seed для train/validation/test; сеть минимизирует ошибку амплитуд после
дифференцируемого одно-FFT оператора Френеля. Лучшая модель выбирается по validation loss.
`initial.pt` нужен для контроля того, чему научилась сеть. `best.pt` — для инференса.


In [ ]:
generate_dataset(DATA, train=counts[0], validation=counts[1], test=counts[2])
training_report = train(DATA, TRAINING, config)
assert training_report['parameter_l2_change_last_vs_initial'] > 0
print('Best validation epoch:', training_report['best_epoch'])


## Независимая тестовая оценка

Сравнение с той же сетью до обучения и многоплоскостным Герчбергом–Сэкстоном
при одинаковом и большем числе оптических шагов. Все методы оцениваются после
квантования фазы и амплитуд в 8 бит. Время отражает batched throughput на текущем
устройстве; ускорение относительно классического метода не предполагается заранее.


In [ ]:
benchmark = evaluate(DATA, TRAINING / 'best.pt', RUN / 'benchmark.json',
                     batch_size=config.batch_size, gs_steps=12 if QUICK_CHECK else 80)
print(json.dumps(benchmark['summary'], indent=2))


## ДОЭ и восстановленные изображения

Сеть выдаёт фазу, кодируемую как `0 → −π`, `255 → +π` с округлением.
Реконструкции рассчитываются заново **из сохранённого PNG ДОЭ**. Сохраняется модуль
комплексного поля, не квадрат модуля. Для отображения каждый модуль делится на свой
максимум; без вычитания фона и без подгонки к целевому изображению.
Метрики считаются по фактическим сохранённым 8-битным PNG. В `.npy` сохранены также
модули до масштабирования и комплексные поля. Шаг выходной сетки зависит от z:
Δx_out = λz/(NΔu); изображения разных плоскостей задаются на соответствующих сетках.


In [ ]:
if INPUT_IMAGES:
    if len(INPUT_IMAGES) != 4:
        raise ValueError('Specify exactly four paths in increasing distance order')
    input_paths = [Path(p) for p in INPUT_IMAGES]
    data_source = 'user-provided quartet'
else:
    input_paths = [DATA / 'test/sample_0000' / f'plane_{i}.png' for i in range(1, 5)]
    data_source = 'synthetic held-out demonstration, not official hackathon inputs'
result = predict(TRAINING / 'best.pt', input_paths, RUN / 'result', source=data_source)
print(json.dumps(result['planes'], indent=2))
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(RUN / 'result/comparison.png')))
display(DisplayImage(filename=str(RUN / 'result/phase_preview.png')))


## Материалы для экспертов

Архив содержит код, полный сгенерированный набор, веса, историю обучения,
тестовое сравнение, входные PNG, фазовый ДОЭ, четыре восстановленных PNG,
метрики и исходные комплексные поля. Качество на официальной четвёрке необходимо
измерить отдельно после её предоставления. Это численное моделирование; оптическая
проверка на ПВМС здесь не выполняется.


In [ ]:
import shutil
submission = RUN / 'submission'
submission.mkdir()
for name, source in SOURCE_FILES.items():
    path = submission / 'source' / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
for name in ('data', 'training', 'result'):
    shutil.copytree(RUN / name, submission / name)
shutil.copy2(RUN / 'benchmark.json', submission / 'benchmark.json')
(submission / 'runtime.json').write_text(json.dumps({'torch': torch.__version__, 'numpy': np.__version__,
    'pillow': PIL.__version__, 'skimage': skimage.__version__, 'device': benchmark['device'],
    'input_source': data_source, 'quick_check': QUICK_CHECK}, indent=2))
archive = shutil.make_archive(str(RUN / 'submission'), 'zip', submission)
print('Submission archive:', archive)
print('Saved results:', RUN / 'result')
